In [ ]:
# SVM Cross-Validation
#
# In this step, a Linear Support Vector Machine (LinearSVC) is evaluated
# using 5-Fold Group Cross-Validation. The subject IDs are used as groups
# so that samples from the same subject never appear in both the training
# and validation folds. This prevents subject-level data leakage and gives
# a more realistic estimate of generalization to unseen subjects.
#
# For each fold, a new LinearSVC model is trained on the training subjects
# and evaluated on completely unseen validation subjects. Accuracy is
# calculated for each fold, followed by the mean and standard deviation
# across all five folds.
#
# The obtained Mean CV Accuracy was 17.64% (Std = 1.45%), which is relatively
# low. The main reasons are:
#
# 1. The task is subject-independent:
#    The model is evaluated on subjects that it has never seen during
#    training. EEG signals vary considerably between subjects, making
#    generalization more difficult.
#
# 2. Linear SVM limitation:
#    LinearSVC learns a linear decision boundary. The relationship between
#    EEG-derived features and emotional states may be nonlinear, so a linear
#    model may not capture all relevant patterns.
#
# 3. Class overlap:
#    The five Valence classes are not perfectly separable in the current
#    feature space. Similar emotional states can produce similar EEG
#    feature patterns.
#
# 4. Feature representation:
#    Although the fused dataset contains 224 features, these features were
#    flattened into a single feature vector. This representation does not
#    explicitly preserve the spatial relationships between EEG channels
#    and therefore does not fully exploit the graph structure of the data.
#
# 5. Baseline purpose:
#    The SVM is used as a classical machine-learning baseline rather than
#    as the final model. Its relatively low performance provides a reference
#    point for evaluating whether the proposed dynamic graph-based model
#    can better exploit spatial and temporal relationships in EEG signals.
#
# Result:
# Mean CV Accuracy = 17.64%
# Std CV Accuracy  = 1.45%

#Note :
# Due to the high computational cost and long training time of the
# machine learning models, especially SVM and cross-validation, the
# remaining experiments were continued using Google Colab to take
# advantage of its available computational resources.


In [16]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:

drive_root = Path("/content/drive/MyDrive")

for path in drive_root.rglob("split_dataset"):
    print(path)

/content/drive/MyDrive/processed_data/split_dataset


In [18]:
import numpy as np
from pathlib import Path

split_dir = Path("/content/drive/MyDrive/processed_data/split_dataset")

X_train = np.load(split_dir / "X_train.npy")
X_val = np.load(split_dir / "X_val.npy")
X_test = np.load(split_dir / "X_test.npy")

y_valence_train = np.load(split_dir / "y_valence_train.npy")
y_valence_val = np.load(split_dir / "y_valence_val.npy")
y_valence_test = np.load(split_dir / "y_valence_test.npy")

subjects_train = np.load(split_dir / "subjects_train.npy")
subjects_val = np.load(split_dir / "subjects_val.npy")
subjects_test = np.load(split_dir / "subjects_test.npy")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("Valence:", y_valence_train.shape, y_valence_val.shape, y_valence_test.shape)

print("Subjects:", subjects_train.shape, subjects_val.shape, subjects_test.shape)

X_train: (59360, 224)
X_val: (11130, 224)
X_test: (14840, 224)
Valence: (59360,) (11130,) (14840,)
Subjects: (59360,) (11130,) (14840,)


In [19]:
from sklearn.svm import SVC

svm_baseline = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    cache_size=4000
)

print("SVM baseline is ready.")

SVM baseline is ready.


In [20]:
from sklearn.svm import LinearSVC

svm_baseline = LinearSVC(
    C=1.0,
    random_state=42,
    max_iter=5000
)

print("Linear SVM baseline is ready.")

Linear SVM baseline is ready.


In [21]:
from sklearn.metrics import accuracy_score, classification_report

In [22]:
print("Training Linear SVM on full training set...")

svm_baseline.fit(
    X_train,
    y_valence_train
)

print("Training completed.")

valence_val_pred_svm = svm_baseline.predict(X_val)

valence_val_accuracy_svm = accuracy_score(
    y_valence_val,
    valence_val_pred_svm
)

print(f"Validation Accuracy: {valence_val_accuracy_svm:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_valence_val,
        valence_val_pred_svm,
        labels=[1, 2, 3, 4, 5],
        digits=4
    )
)

Training Linear SVM on full training set...
Training completed.
Validation Accuracy: 0.2144

Classification Report:
              precision    recall  f1-score   support

           1     0.2104    0.3187    0.2535      1939
           2     0.3769    0.3598    0.3682      3035
           3     0.0647    0.0734    0.0687      1963
           4     0.1629    0.1301    0.1447      2574
           5     0.1945    0.1217    0.1497      1619

    accuracy                         0.2144     11130
   macro avg     0.2019    0.2007    0.1970     11130
weighted avg     0.2168    0.2144    0.2119     11130



In [23]:
from pathlib import Path

models_dir = Path("/content/drive/MyDrive/models")
models_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Models directory:", models_dir)

Models directory: /content/drive/MyDrive/models


In [24]:
import joblib

svm_path = models_dir / "linear_svm_valence_baseline.joblib"

joblib.dump(
    svm_baseline,
    svm_path
)

print("Linear SVM model saved.")
print("Path:", svm_path.resolve())

Linear SVM model saved.
Path: /content/drive/MyDrive/models/linear_svm_valence_baseline.joblib


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score
import numpy as np

group_kfold = GroupKFold(n_splits=5)

svm_cv_scores = []

for fold, (train_idx, val_idx) in enumerate(
    group_kfold.split(X_train, y_valence_train, groups=subjects_train),
    start=1
):
    X_fold_train = X_train[train_idx]
    X_fold_val = X_train[val_idx]

    y_fold_train = y_valence_train[train_idx]
    y_fold_val = y_valence_train[val_idx]

    subjects_fold_train = subjects_train[train_idx]
    subjects_fold_val = subjects_train[val_idx]

    svm_fold = LinearSVC(
        C=1.0,
        random_state=42,
        max_iter=5000
    )

    svm_fold.fit(
        X_fold_train,
        y_fold_train
    )

    fold_pred = svm_fold.predict(X_fold_val)
    fold_accuracy = accuracy_score(
        y_fold_val,
        fold_pred
    )

    svm_cv_scores.append(fold_accuracy)

    print(
        f"Fold {fold} Accuracy: {fold_accuracy:.4f}"
    )

    print(
        f"Train subjects: {np.unique(subjects_fold_train)}"
    )

    print(
        f"Validation subjects: {np.unique(subjects_fold_val)}"
    )

print(
    f"\nMean CV Accuracy: {np.mean(svm_cv_scores):.4f}"
)

print(
    f"Std CV Accuracy: {np.std(svm_cv_scores):.4f}"
)